# Machine Learning with RustQuant

RustQuant includes machine learning tools commonly used in quantitative finance:

- **Linear Regression** (OLS with QR/SVD decomposition)
- **Ridge Regression** (L2 regularization)
- **Lasso Regression** (L1 regularization)
- **Logistic Regression** (binary classification)
- **K-Nearest Neighbors** (classification)

## Setup

In [ ]:
:dep RustQuant = { path = "../crates/RustQuant" }
:dep nalgebra = "0.33"

## 1. Linear Regression

Ordinary Least Squares regression using matrix decomposition.
Supports QR and SVD decomposition methods.

In [ ]:
use nalgebra::{DMatrix, DVector};
use RustQuant::ml::*;

// Training data: 4 samples, 3 features
let x_train = DMatrix::from_row_slice(
    4, 3,
    &[
        -0.084, -0.633, -0.399,
        -0.983,  1.091, -0.468,
        -1.875, -0.914,  0.327,
        -0.186,  1.002, -0.413,
    ],
);

// Test data: 4 samples, 3 features
let x_test = DMatrix::from_row_slice(
    4, 3,
    &[
        0.562, 0.596, -0.412,
        0.663, 0.452, -0.294,
       -0.603, 0.897,  1.219,
        0.698, 0.572,  0.244,
    ],
);

let y_train = DVector::from_row_slice(&[-0.445, -1.848, -0.629, -0.861]);

let input = LinearRegressionInput {
    x: x_train,
    y: y_train,
};

// Fit using QR decomposition
let output = input.fit(Decomposition::QR).unwrap();
let predictions = output.predict(x_test).unwrap();

println!("=== Linear Regression (QR) ===");
println!("Intercept:    {:.4}", output.intercept);
println!("Coefficients: {:?}", output.coefficients.iter().map(|c| format!("{:.4}", c)).collect::<Vec<_>>());
println!("Predictions:  {:?}", predictions.iter().map(|p| format!("{:.4}", p)).collect::<Vec<_>>());

## 2. Comparing Decomposition Methods

Both QR and SVD should give the same result for well-conditioned problems.

In [ ]:
let x_train = DMatrix::from_row_slice(
    4, 3,
    &[
        -0.084, -0.633, -0.399,
        -0.983,  1.091, -0.468,
        -1.875, -0.914,  0.327,
        -0.186,  1.002, -0.413,
    ],
);
let y_train = DVector::from_row_slice(&[-0.445, -1.848, -0.629, -0.861]);

let input_qr = LinearRegressionInput { x: x_train.clone(), y: y_train.clone() };
let input_svd = LinearRegressionInput { x: x_train, y: y_train };

let qr_result = input_qr.fit(Decomposition::QR).unwrap();
let svd_result = input_svd.fit(Decomposition::SVD).unwrap();

println!("{:<12} {:<15} {:<15}", "Param", "QR", "SVD");
println!("{}", "-".repeat(42));
println!("{:<12} {:<15.6} {:<15.6}", "Intercept", qr_result.intercept, svd_result.intercept);
for i in 0..3 {
    println!("{:<12} {:<15.6} {:<15.6}", 
        format!("Beta_{}", i + 1), 
        qr_result.coefficients[i], 
        svd_result.coefficients[i]
    );
}

## 3. Logistic Regression

Binary classification using logistic regression with Iteratively Reweighted Least Squares (IRLS).

In [ ]:
// Generate a simple 2D classification dataset
let x_train = DMatrix::from_row_slice(
    8, 2,
    &[
        // Class 0 (negative examples)
        -2.0, -1.0,
        -1.5, -0.5,
        -1.0, -1.5,
        -0.5, -0.8,
        // Class 1 (positive examples)
         1.0,  0.5,
         1.5,  1.0,
         2.0,  1.5,
         0.5,  1.2,
    ],
);

let y_train = DVector::from_row_slice(&[0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0]);

let input = LogisticRegressionInput {
    x: x_train,
    y: y_train,
};

// Fit using IRLS (Iteratively Reweighted Least Squares)
let output = input.fit(
    LogisticRegressionAlgorithm::IRLS,
    f64::EPSILON.sqrt(),  // convergence tolerance
).unwrap();

println!("=== Logistic Regression (IRLS) ===");
println!("Iterations:   {}", output.iterations);
println!("Coefficients: {:?}", output.coefficients.iter().map(|c| format!("{:.4}", c)).collect::<Vec<_>>());
println!("  (first coefficient is the intercept)");

// Predict on new data
let x_test = DMatrix::from_row_slice(
    4, 2,
    &[
        -1.0, -1.0,   // should be class 0
         0.0,  0.0,   // borderline
         1.0,  1.0,   // should be class 1
         2.0,  2.0,   // should be class 1
    ],
);

let probabilities = output.predict_proba(&x_test);
let predictions = output.predict(&x_test);
println!("\nPredictions:");
for i in 0..4 {
    println!("  Sample {}: P(class=1) = {:.4}, predicted = {}", i + 1, probabilities[i], predictions[i]);
}

## 4. Activation Functions

RustQuant provides common activation functions via the `ActivationFunction` trait,
implemented for `f64`, `Variable` (autodiff), and `DVector<f64>`.

In [ ]:
use RustQuant::ml::ActivationFunction;

let x_values = vec![-2.0_f64, -1.0, 0.0, 1.0, 2.0];

println!("{:<6} {:<12} {:<12} {:<12} {:<12}", "x", "Sigmoid", "Tanh", "ReLU", "Softplus");
println!("{}", "-".repeat(54));

for x in &x_values {
    println!("{:<6.1} {:<12.6} {:<12.6} {:<12.6} {:<12.6}",
        x,
        x.sigmoid(),
        x.tanh(),
        x.relu(),
        x.softplus(),
    );
}

## Summary

| Model | Type | Key Features |
|-------|------|--------------|
| Linear Regression | Regression | QR/SVD decomposition |
| Ridge Regression | Regression | L2 regularization |
| Lasso Regression | Regression | L1 regularization |
| Logistic Regression | Classification | IRLS optimization |
| K-Nearest Neighbors | Classification | Distance-based |

All models use the `nalgebra` crate for efficient linear algebra operations.